In [0]:
# Silver Layer: Shipments — Cleansing + CDC MERGE (BATCH ITERATOR)

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, upper, try_to_date, to_timestamp,
    when, coalesce, lit, current_timestamp,
    row_number, desc
)
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE = "ecomstore.ecomlake.bronze_shipments"
SILVER_TABLE = "ecomstore.ecomlake.silver_shipments"
QUARANTINE_TABLE = "ecomstore.ecomlake.silver_quarantine_shipments"
TEMP_STAGING_TABLE = "ecomstore.ecomlake.temp_shipments_staging"

bronze_df = spark.read.table(BRONZE_TABLE)

# 1. Fetch batches chronologically
batches_df = spark.sql(f"SELECT DISTINCT _batch_id FROM {BRONZE_TABLE} ORDER BY _batch_id").collect()
batches = [row['_batch_id'] for row in batches_df]

for current_batch in batches:
    print(f"\n🚀 Processing Batch: {current_batch}")
    
    incoming_batch = bronze_df.filter(col("_batch_id") == current_batch)

    # 2. Deduplication WITHIN the current batch
    w = Window.partitionBy("shipment_id").orderBy(desc("updated_at"), desc("_ingestion_timestamp"))
    deduped_df = incoming_batch.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn") 

    # 3. Cleansing & Standardization
    cleaned_df = (
        deduped_df
        .withColumn("shipment_id", trim(upper(col("shipment_id"))))
        .withColumn("order_id",    trim(upper(col("order_id"))))
        .withColumn("carrier",     trim(col("carrier")))
        .withColumn("warehouse_id", trim(upper(col("warehouse_id"))))
        .withColumn("shipment_status",
            when(col("shipment_status").isin("dispatched", "in_transit", "out_for_delivery", "delivered", "lost"), col("shipment_status")).otherwise(lit("unknown"))
        )
        .withColumn("dispatch_date", coalesce(try_to_date(col("dispatch_date"), "yyyy-MM-dd"), try_to_date(col("dispatch_date"), "dd/MM/yyyy")))
        .withColumn("promised_delivery_date", coalesce(try_to_date(col("promised_delivery_date"), "yyyy-MM-dd"), try_to_date(col("promised_delivery_date"), "dd/MM/yyyy")))
        .withColumn("actual_delivery_date", coalesce(try_to_date(col("actual_delivery_date"), "yyyy-MM-dd"), try_to_date(col("actual_delivery_date"), "dd/MM/yyyy")))
        .withColumn("delivery_attempts",
            when(col("delivery_attempts").cast(DoubleType()).cast(IntegerType()) > 0, col("delivery_attempts").cast(DoubleType()).cast(IntegerType())).otherwise(None)
        )
        .withColumn("created_at", coalesce(to_timestamp(col("created_at")), to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss")))
        .withColumn("updated_at", coalesce(to_timestamp(col("updated_at")), to_timestamp(col("updated_at"), "yyyy-MM-dd HH:mm:ss")))
        .withColumn("_silver_processed_at", current_timestamp())
    )

    silver_schema_cols = [
        "shipment_id", "order_id", "carrier", "tracking_number",
        "shipment_status", "dispatch_date", "promised_delivery_date",
        "actual_delivery_date", "delivery_attempts", "warehouse_id",
        "created_at", "updated_at", "_batch_id", "_silver_processed_at"
    ]

    # Disk Staging
    cleaned_df.select(*silver_schema_cols).coalesce(1).write.format("delta").mode("overwrite").saveAsTable(TEMP_STAGING_TABLE)
    staged_silver_df = spark.read.table(TEMP_STAGING_TABLE)

    # 4. Data Quality split
    good_records = staged_silver_df.filter(col("shipment_id").isNotNull() & col("order_id").isNotNull() & col("dispatch_date").isNotNull())
    bad_records = staged_silver_df.filter(col("shipment_id").isNull() | col("order_id").isNull() | col("dispatch_date").isNull())

    if bad_records.count() > 0:
        (
            bad_records
            .withColumn("rejection_reason",
                when(col("shipment_id").isNull(), lit("null_shipment_id"))
                .when(col("order_id").isNull(), lit("null_order_id"))
                .otherwise(lit("null_dispatch_date"))
            )
            .coalesce(1)
            .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(QUARANTINE_TABLE)
        )

    # 5. CDC MERGE
    if spark.catalog.tableExists(SILVER_TABLE):
        silver_target = DeltaTable.forName(spark, SILVER_TABLE)
        (
            silver_target.alias("target")
            .merge(good_records.alias("source"), "target.shipment_id = source.shipment_id")
            .whenMatchedUpdate(
                condition="source.updated_at > target.updated_at",
                set={c: f"source.{c}" for c in silver_schema_cols if c not in ["shipment_id", "created_at"]}
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"✅ CDC MERGE complete for Batch: {current_batch}")
    else:
        good_records.coalesce(1).write.format("delta").mode("overwrite").option("delta.autoOptimize.optimizeWrite", "true").option("delta.autoOptimize.autoCompact", "true").saveAsTable(SILVER_TABLE)
        print(f"✅ Created initial shipments table for Batch: {current_batch}")

spark.sql(f"DROP TABLE IF EXISTS {TEMP_STAGING_TABLE}")